# How to apply penalised logistic regression, explainable boosting machine, and xgboost models to predict a binary outcome


This script contains minimal code for applying these types of models to the data,
including the finding of best hyperparameteres.

We first split data into train (80%), validation (20%) and test (20%) sets.
We fit models with different hyperparameters on train set, evaluate them on validation set using the AUC metric,
then pick the hyperparameter combination that had the highest validation set AUC.

If the validation set is small, or has a small number of individuals with an outcome.
then this is not the most rigorous way of finding hyperparameters.
In that case, the validation set scores may not be representative,
as they are driven by the smaller number of individuals with the outcome.
A more rigorous way is to use an inner cross-validation loop:
- split out the test set
- split the remaining data into four folds (as an example)
- at each iteration, one fold serves as the validation set and the remaining as the train set
  we train the model with a hyperparameter combination on each of the four training sets,
  and evaluate it on the corresponding validation set.
  For each hyperparameter combination, we thus have four models we trained and four validation set scores.
  We compute the average validation set score, and perhaps its standard deviation.
  We select the combination that had highest average validation set score,
However, this can be time consuming in practice.
For simplicity, we keep a single train-validation split.

In [ ]:
# Install necessary python packages
%pip install pandas==2.2.3
%pip install numpy==2.0.2
%pip install scikit-learn==1.4.2
%pip install interpret==0.7.3
%pip install xgboost==2.1.4
%pip install optuna==5.0.0

In [ ]:
# Load packages
from sklearn.linear_model import LogisticRegression   # for logistic regression
from sklearn.model_selection import train_test_split  # to split data into train-validation-test sets
from sklearn.metrics import roc_auc_score             # to evaluate how the models perform
from sklearn.preprocessing import StandardScaler      # to standardise values of input data to zero mean and unit standard deviation
from interpret.glassbox import ExplainableBoostingClassifier  # for explainable boosting machine (EBM)
from interpret import show         # to visualise shape functions of EBM
from xgboost import XGBClassifier  # for XGBoost
import optuna        # for hyperparameter tuning
import pandas as pd  # for working with data tables
import numpy as np

/Users/andres/opt/anaconda3/envs/clinpred/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Prepare Data

In [2]:
# Helper function to create a dummy dataset
def dummy_data(n=1000, bias=-6, random_state=42,
               w_plt=3., w_age=4.5, w_sex=1., p_noise=30):

    # Dummy predictor variables
    rng = np.random.default_rng(seed=random_state)
    #fit = rng.lognormal(mean=1.2, sigma=1.75, size=n).clip(0, 400)
    ind_male = rng.binomial(n=1, p=0.4, size=n)
    platelets = rng.gamma(shape=1.2, scale=170, size=n).clip(50, 800).round(1)
    age = rng.normal(loc=70, scale=10, size=n).clip(18, 100).round(1)

    # Transform the predictor variables by their respective functions
    f_age = -1 * ((age - 85) / 10)**2
    f_plt = ((platelets - 400) / 100)**2
    f_male = w_sex * ind_male

    # scale to [0, 1] * weight
    f_age = w_age * (f_age - f_age.min()) / (f_age.max() - f_age.min())
    f_plt = w_plt * (f_plt - f_plt.min()) / (f_plt.max() - f_plt.min())

    # Get additive predictor
    m = bias + f_age + f_plt + f_male
    prob = 1 / (1 + np.exp(-m))
    crc = rng.binomial(n=1, p=prob, size=n)

    # Collect to dataframe
    df = pd.DataFrame({'cancer': crc,
                       'age': age,
                       'sex_male': ind_male,
                       'platelets': platelets})
    print(f"{n} observations, {crc.sum()} cancers, {crc.mean()*100}% cancers")
    shapes = pd.DataFrame({'f_age': f_age, 'f_male': f_male, 'f_platelets': f_plt})

    # Add noise variables
    x_noise = rng.uniform(-1, 1, size=(n, p_noise))
    x_noise = pd.DataFrame(x_noise)
    x_noise.columns = ['noise' + str(i) for i, c in enumerate(x_noise.columns)]
    df = pd.concat(objs=[df, x_noise], axis=1)

    return df, shapes

In [3]:
# Create simulated data
#  The data will have three variables that are associated with the cancer outcome: age, sex, platelets
#  it will also have 100 variables that have no association, these are called "noise1", "noise2", etc
#  When the noise variables are included, the models are more likely to overfit, as they will try to use
#  the random patterns that appear in sampled noise variables to classify patients, even thouhgh these patterns
#  are not generalisable
df, __ = dummy_data(n=1000, p_noise=100, bias=-6, w_age=5., w_plt=4.8)
print(df.shape[0], "rows in data")
print(df.shape[1], "columns in data")
print(df.cancer.sum(), "num individuals with cancer")

1000 observations, 611 cancers, 61.1% cancers
1000 rows in data
104 columns in data
611 num individuals with cancer


In [4]:
# Divide the data into an outcome variable (y) and predictor variables (x)
x = df.drop(labels=['cancer'], axis=1)
y = df.cancer

In [5]:
# Split data into "model selection (sel) and test sets, setting aside 20% as the test set
# We use a stratified split, so that the proportion of patients with cancer remains constant
x_sel, x_test, y_sel, y_test = train_test_split(x, y, stratify=y, test_size=0.2, random_state=42)

In [6]:
# Split model selection set further into train and validation (val) sets, so that val set forms 20% of the total data
x_train, x_val, y_train, y_val = train_test_split(x_sel, y_sel, stratify=y_sel, test_size=0.25, random_state=42)

In [7]:
# Review number of observations in the datasets, and number of people with cancer ine ach
print(y_train.shape[0], y_val.shape[0], y_test.shape[0])
print(y_train.sum(), y_val.sum(), y_test.sum())

600 200 200
367 122 122


In [8]:
# Scale the input data (only relevant for logistic regression, as it helps the optimisation algorithm to converge faster)
scaler = StandardScaler()
scaler.fit(x_train)
x_train_tf = scaler.transform(x_train)
x_val_tf = scaler.transform(x_val)

scaler_sel = StandardScaler()
scaler_sel.fit(x_sel)
x_sel_tf = scaler_sel.transform(x_sel)
x_test_tf = scaler_sel.transform(x_test)

## Logistic Regression

In [9]:
# Fit a regular logistic model on the model selection set and see how it performs on test set
# In that case we don't need the train + val split, as there's no hyperparameters to tune
# so our model selection set IS the train set
clf = LogisticRegression(penalty=None)
clf.fit(x_sel_tf, y_sel)

pred_sel = clf.predict_proba(x_sel_tf)[:, 1]
pred_test = clf.predict_proba(x_test_tf)[:, 1]

auc_sel = roc_auc_score(y_sel, pred_sel)
auc_test = roc_auc_score(y_test, pred_test)
print("auc train+val:", auc_sel)  # see how the auc on training set is higher (0.8) than on test set (0.69), indicating overfitting
print("auc test", auc_test)

auc train+val: 0.8078038387943108
auc test 0.6927280369903321


In [10]:
# Look at regression coefficients
coef = clf.coef_
coef = pd.DataFrame(coef, columns=x_train.columns)
coef = coef.transpose().reset_index()
coef.columns = ['variable', 'coef']
coef['coef_abs'] = coef.coef.abs()
coef = coef.sort_values(by=['coef_abs'], ascending=False).reset_index(drop=True)
print("Top 10 highest coefficients:")
print((coef.coef == 0).sum(), "of", coef.shape[0], "coefficients are zero")
coef.iloc[0:10]


Top 10 highest coefficients:
0 of 103 coefficients are zero


,variable,coef,coef_abs
0,platelets,-0.676998,0.676998
1,age,0.636432,0.636432
2,sex_male,0.304173,0.304173
3,noise79,0.226435,0.226435
4,noise25,0.202237,0.202237
5,noise61,-0.196424,0.196424
6,noise19,-0.194464,0.194464
7,noise0,-0.183918,0.183918
8,noise7,-0.179116,0.179116
9,noise67,-0.159785,0.159785


## Penalised Logistic Regression (Lasso)

In [11]:
# Fit penalised logistic regression with Lasso penalty and default penalty strength value of 1
clf = LogisticRegression(penalty="l1", solver="saga", C=1, max_iter=10000)
clf.fit(x_sel_tf, y_sel)

pred_sel = clf.predict_proba(x_sel_tf)[:, 1]
pred_test = clf.predict_proba(x_test_tf)[:, 1]

auc_sel = roc_auc_score(y_sel, pred_sel)
auc_test = roc_auc_score(y_test, pred_test)
print("auc train+val:", auc_sel)
print("auc test", auc_test)

auc train+val: 0.8086915353204585
auc test 0.7008196721311476


In [12]:
# Look at regression coefficients
coef = clf.coef_
coef = pd.DataFrame(coef, columns=x_train.columns)
coef = coef.transpose().reset_index()
coef.columns = ['variable', 'coef']
coef['coef_abs'] = coef.coef.abs()
coef = coef.sort_values(by=['coef_abs'], ascending=False).reset_index(drop=True)
print("Top 10 highest coefficients:")
print((coef.coef == 0).sum(), "of", coef.shape[0], "coefficients are zero")
coef.iloc[0:10]

Top 10 highest coefficients:
9 of 103 coefficients are zero


,variable,coef,coef_abs
0,platelets,-0.653762,0.653762
1,age,0.614966,0.614966
2,sex_male,0.288214,0.288214
3,noise79,0.213807,0.213807
4,noise61,-0.192466,0.192466
5,noise25,0.186302,0.186302
6,noise19,-0.181387,0.181387
7,noise0,-0.170770,0.170770
8,noise7,-0.163907,0.163907
9,noise67,-0.142926,0.142926


In [13]:
# Now, let's tune penalty strength
# Try different values for penalty strength lambda (note in sklearn, the parameter "C" is inverse of lambda)
result = pd.DataFrame()
lambdas = [0.001, 0.01, 0.1, 1, 10, 100, 1000]
for lam in lambdas:
    print("penalty strength", lam)

    clf = LogisticRegression(penalty='l1', C=1/lam, max_iter=10000, solver='saga')
    clf.fit(x_train_tf, y_train)

    pred_train = clf.predict_proba(x_train_tf)[:, 1]
    pred_val = clf.predict_proba(x_val_tf)[:, 1]

    auc_train = roc_auc_score(y_train, pred_train)
    auc_val = roc_auc_score(y_val, pred_val)
    print("auc train:", auc_train, "auc val", auc_val)

    result_lam = {'lam': lam, "auc_train": auc_train, "auc_val": auc_val}
    result_lam = pd.DataFrame(result_lam, index=[0])
    result = pd.concat(objs=[result, result_lam], axis=0)

penalty strength 0.001
auc train: 0.8214849551519687 auc val 0.692728036990332
penalty strength 0.01
auc train: 0.8215083439557485 auc val 0.692833123160992
penalty strength 0.1
auc train: 0.821473260750079 auc val 0.6931483816729719
penalty strength 1
auc train: 0.8226427009390604 auc val 0.6997688104245481
penalty strength 10
auc train: 0.8127609313421665 auc val 0.7679697351828499
penalty strength 100
auc train: 0.5 auc val 0.5
penalty strength 1000
auc train: 0.5 auc val 0.5


In [14]:
# Look at the results
# In this case, the penalties perform similarly, although lambda=0.1 achieves a slightly higher validation set score
result = result.reset_index(drop=True)
result

,lam,auc_train,auc_val
0,0.001,0.821485,0.692728
1,0.010,0.821508,0.692833
2,0.100,0.821473,0.693148
3,1.000,0.822643,0.699769
4,10.000,0.812761,0.767970
5,100.000,0.500000,0.500000
6,1000.000,0.500000,0.500000


In [15]:
# Refit the model with best lambda on model selection set (train+val), evaluate on test set
lam = result.loc[result.auc_val.idxmax(), 'lam']
clf = LogisticRegression(penalty='l1', C=1/lam, max_iter=10000, solver='saga')
clf.fit(x_sel_tf, y_sel)

pred_sel = clf.predict_proba(x_sel_tf)[:, 1]
pred_test = clf.predict_proba(x_test_tf)[:, 1]

auc_sel = roc_auc_score(y_sel, pred_sel)
auc_test = roc_auc_score(y_test, pred_test)
print("auc train+val:", auc_sel)
print("auc test", auc_test)

auc train+val: 0.8028853424864708
auc test 0.7428541403951241


In [16]:
# Look at regression coefficients
coef = clf.coef_
coef = pd.DataFrame(coef, columns=x_train.columns)
coef = coef.transpose().reset_index()
coef.columns = ['variable', 'coef']
coef['coef_abs'] = coef.coef.abs()
coef = coef.sort_values(by=['coef_abs'], ascending=False).reset_index(drop=True)
print("Top 10 highest coefficients:")
print((coef.coef == 0).sum(), "of", coef.shape[0], "coefficients are zero")
coef.iloc[0:10]

Top 10 highest coefficients:
61 of 103 coefficients are zero


,variable,coef,coef_abs
0,platelets,-0.523606,0.523606
1,age,0.496845,0.496845
2,sex_male,0.208078,0.208078
3,noise61,-0.154454,0.154454
4,noise79,0.135441,0.135441
5,noise19,-0.108386,0.108386
6,noise0,-0.104412,0.104412
7,noise25,0.085640,0.085640
8,noise84,0.072271,0.072271
9,noise7,-0.067655,0.067655


## Penalised Logistic Regression: Elastic Net

In [17]:
# Fit elastic net with penalty strength 1 and alpha 0.5 (both lasso and ridge penalties get a weight of 0.5)
clf = LogisticRegression(penalty="elasticnet", solver="saga", C=1, l1_ratio=0.5, max_iter=10000)
clf.fit(x_sel_tf, y_sel)

pred_sel = clf.predict_proba(x_sel_tf)[:, 1]
pred_test = clf.predict_proba(x_test_tf)[:, 1]

auc_sel = roc_auc_score(y_sel, pred_sel)
auc_test = roc_auc_score(y_test, pred_test)
print("auc train+val:", auc_sel)
print("auc test", auc_test)

auc train+val: 0.8082180971731798
auc test 0.6970365699873896


In [18]:
# Look at regression coefficients
coef = clf.coef_
coef = pd.DataFrame(coef, columns=x_train.columns)
coef = coef.transpose().reset_index()
coef.columns = ['variable', 'coef']
coef['coef_abs'] = coef.coef.abs()
coef = coef.sort_values(by=['coef_abs'], ascending=False).reset_index(drop=True)
print("Top 10 highest coefficients:")
print((coef.coef == 0).sum(), "of", coef.shape[0], "coefficients are zero")
coef.iloc[0:10]

Top 10 highest coefficients:
4 of 103 coefficients are zero


,variable,coef,coef_abs
0,platelets,-0.661184,0.661184
1,age,0.622141,0.622141
2,sex_male,0.293851,0.293851
3,noise79,0.218993,0.218993
4,noise61,-0.193888,0.193888
5,noise25,0.192804,0.192804
6,noise19,-0.186581,0.186581
7,noise0,-0.176161,0.176161
8,noise7,-0.170006,0.170006
9,noise67,-0.150176,0.150176


In [19]:
# Let's tune parameters of elastic net using sequential model-based optimisation
def objective(trial):
    C = trial.suggest_float("C", 1e-5, 1e5, log=True)
    l1_ratio = trial.suggest_float("l1_ratio", 0, 1)
    clf = LogisticRegression(penalty='elasticnet', C=C, l1_ratio=l1_ratio, max_iter=10000, solver='saga')
    clf.fit(x_train_tf, y_train)
    pred_val = clf.predict_proba(x_val_tf)[:, 1]
    return roc_auc_score(y_val, pred_val)  # validation AUC

In [20]:
# Run the hyperparameter search
study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=100)
print("Best hyperparameters:", study.best_params)
print("Best penalty strength lambda (1/C):", 1 / study.best_params["C"])
print("Best validation AUC:", study.best_value)

[I 2026-11-08 17:01:51,386] A new study created in memory with name: no-name-34e0be44-8b05-41eb-b84d-eaa596395c56
[I 2026-11-08 17:01:51,409] Trial 0 finished with value: 0.7867801597309794 and parameters: {'C': 0.05564180225431373, 'l1_ratio': 0.9507143064099162}. Best is trial 0 with value: 0.7867801597309794.
[I 2026-11-08 17:01:51,434] Trial 1 finished with value: 0.692728036990332 and parameters: {'C': 208.90047049266641, 'l1_ratio': 0.5986584841970366}. Best is trial 0 with value: 0.7867801597309794.
[I 2026-11-08 17:01:51,440] Trial 2 finished with value: 0.5 and parameters: {'C': 0.0003632339256943143, 'l1_ratio': 0.15599452033620265}. Best is trial 0 with value: 0.7867801597309794.
[I 2026-11-08 17:01:51,446] Trial 3 finished with value: 0.5 and parameters: {'C': 3.809220577048033e-05, 'l1_ratio': 0.8661761457749352}. Best is trial 0 with value: 0.7867801597309794.
[I 2026-11-08 17:01:51,473] Trial 4 finished with value: 0.692728036990332 and parameters: {'C': 10.2600651248967

Best hyperparameters: {'C': 0.05564180225431373, 'l1_ratio': 0.9507143064099162}
Best penalty strength lambda (1/C): 17.972099383651315
Best validation AUC: 0.7867801597309794


In [21]:
# Refit elastic net with the best hyperparameters on model selection set (train+val), evaluate on test set
# The test set is scaled with the scaler fitted on the model selection set, to match the data the model is fitted on
clf = LogisticRegression(penalty='elasticnet', **study.best_params, max_iter=10000, solver='saga')
clf.fit(x_sel_tf, y_sel)

pred_sel = clf.predict_proba(x_sel_tf)[:, 1]
pred_test = clf.predict_proba(x_test_tf)[:, 1]

auc_sel = roc_auc_score(y_sel, pred_sel)
auc_test = roc_auc_score(y_test, pred_test)
print(f"auc sel: {auc_sel}, auc test: {auc_test}")

auc sel: 0.7944555132529802, auc test: 0.7491593106347204


In [22]:
# Look at regression coefficients (some may be exactly zero due to the L1 part of the penalty)
coef = clf.coef_
coef = pd.DataFrame(coef, columns=x_train.columns)
coef = coef.transpose().reset_index()
coef.columns = ['variable', 'coef']
coef['coef_abs'] = coef.coef.abs()
coef = coef.sort_values(by=['coef_abs'], ascending=False).reset_index(drop=True)
print("Top 10 highest coefficients:")
print((coef.coef == 0).sum(), "of", coef.shape[0], "coefficients are zero")
coef.iloc[0:10]

Top 10 highest coefficients:
83 of 103 coefficients are zero


,variable,coef,coef_abs
0,platelets,-0.470030,0.470030
1,age,0.437432,0.437432
2,sex_male,0.163602,0.163602
3,noise61,-0.119383,0.119383
4,noise79,0.092461,0.092461
5,noise0,-0.066595,0.066595
6,noise19,-0.061051,0.061051
7,noise25,0.040928,0.040928
8,noise84,0.028980,0.028980
9,noise27,-0.020490,0.020490


## Explainable Boosting Machine

In [23]:
# Fit EBM with default settings on train+val data
# EBM does not require data to be scaled beforehand (i.e. it does not improve optimisation speed)
# It may nevertheless run slower than logistic regression and XGBoost.
# Set number of interactions to 10 to speed it up
clf = ExplainableBoostingClassifier(interactions=10)
clf.fit(x_sel, y_sel)

pred_sel = clf.predict_proba(x_sel)[:, 1]
pred_test = clf.predict_proba(x_test)[:, 1]

auc_sel = roc_auc_score(y_sel, pred_sel)
auc_test = roc_auc_score(y_test, pred_test)
print("auc train+val:", auc_sel)
print("auc test", auc_test)

auc train+val: 0.974552699583769
auc test 0.7668137873055906


In [24]:
# Visualise shape functions (may need to open the link in broswer)
# Note how some effects that are identified are real (e.g. platelets and age)
# and how some are due to overfitting (e.g. interaction between platelets and noise variable 78)
show(clf.explain_global())

<!-- http://127.0.0.1:7014/4928873040/ -->

In [25]:
# Look at feature importances
coef = clf.term_importances().reshape(1, -1)
coef = pd.DataFrame(coef, columns=clf.term_names_)
coef = coef.transpose().reset_index()
coef.columns = ['variable', 'importance']
coef = coef.sort_values(by=['importance'], ascending=False).reset_index(drop=True)
print((coef.importance == 0).sum(), "of", coef.shape[0], "coefficients are zero")
print("Top 10 most important variables:")
coef.iloc[0:10]

0 of 113 coefficients are zero
Top 10 most important variables:


,variable,importance
0,platelets,0.917623
1,age,0.477749
2,platelets & noise78,0.218888
3,sex_male,0.161798
4,noise19 & noise50,0.130427
5,age & noise7,0.119819
6,noise21 & noise39,0.115376
7,noise23 & noise84,0.114570
8,age & noise5,0.111864
9,noise61,0.102187


In [26]:
# Use sequential model optimisation to find best hyperparameters for EBM
def objective(trial):
    params = {
        "max_bins": trial.suggest_categorical("max_bins", [128, 1024]),
        "max_interaction_bins": trial.suggest_categorical("max_interaction_bins", [16, 64]),
        "interactions": trial.suggest_categorical("interactions", [0, 10]),
        "reg_alpha": trial.suggest_categorical("reg_alpha", [0, 0.01, 0.1, 1, 10]),
        "reg_lambda": trial.suggest_categorical("reg_lambda", [0, 0.01, 0.1, 1, 10]),
        "max_leaves": trial.suggest_categorical("max_leaves", [2, 3]),
        "learning_rate": trial.suggest_float("learning_rate", 0.0025, 0.2, log=True),
        "validation_size": trial.suggest_categorical("validation_size", [0.15, 0.3]),
    }
    clf = ExplainableBoostingClassifier(**params)
    clf.fit(x_train, y_train)
    pred_val = clf.predict_proba(x_val)[:, 1]
    return roc_auc_score(y_val, pred_val)  # validation AUC

In [27]:
# Run the hyperparameter search
# EBM is slower to fit than XGBoost, so fewer trials are used; increase n_trials if time allows
study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=50)
best_params = study.best_params
print("Best hyperparameters:", best_params)
print("Best validation AUC:", study.best_value)

[I 2026-11-08 17:02:18,226] A new study created in memory with name: no-name-ed144731-1569-418a-9d5a-474f88abcd97
[I 2026-11-08 17:02:23,883] Trial 0 finished with value: 0.8366960907944514 and parameters: {'max_bins': 1024, 'max_interaction_bins': 16, 'interactions': 0, 'reg_alpha': 0.01, 'reg_lambda': 0, 'max_leaves': 3, 'learning_rate': 0.01659473221041593, 'validation_size': 0.3}. Best is trial 0 with value: 0.8366960907944514.
[I 2026-11-08 17:02:28,368] Trial 1 finished with value: 0.8441572089113072 and parameters: {'max_bins': 1024, 'max_interaction_bins': 64, 'interactions': 0, 'reg_alpha': 1, 'reg_lambda': 0.1, 'max_leaves': 3, 'learning_rate': 0.01720243051347517, 'validation_size': 0.3}. Best is trial 1 with value: 0.8441572089113072.
[I 2026-11-08 17:02:42,838] Trial 2 finished with value: 0.8269230769230769 and parameters: {'max_bins': 1024, 'max_interaction_bins': 64, 'interactions': 10, 'reg_alpha': 0.1, 'reg_lambda': 0.1, 'max_leaves': 3, 'learning_rate': 0.01372865106

Best hyperparameters: {'max_bins': 128, 'max_interaction_bins': 64, 'interactions': 0, 'reg_alpha': 0.01, 'reg_lambda': 1, 'max_leaves': 3, 'learning_rate': 0.00575030855694562, 'validation_size': 0.3}
Best validation AUC: 0.8557166876839007


In [28]:
# Refit EBM with the best hyperparameters on train + val set
clf = ExplainableBoostingClassifier(**best_params)
clf.fit(x_sel, y_sel)

# Make predictions on training+validation and test data
pred_sel = clf.predict_proba(x_sel)[:, 1]
pred_test = clf.predict_proba(x_test)[:, 1]

# Evaluate
auc_sel = roc_auc_score(y_sel, pred_sel)
auc_test = roc_auc_score(y_test, pred_test)
print(f"auc sel: {auc_sel}, auc test: {auc_test}")

auc sel: 0.9307925486096041, auc test: 0.7763766288356452


In [29]:
# Visualise shape functions of the tuned model
show(clf.explain_global())

<!-- http://127.0.0.1:7014/5009465744/ -->

In [30]:
# Look at feature importances
coef = clf.term_importances().reshape(1, -1)
coef = pd.DataFrame(coef, columns=clf.term_names_)
coef = coef.transpose().reset_index()
coef.columns = ['variable', 'importance']
coef = coef.sort_values(by=['importance'], ascending=False).reset_index(drop=True)
print((coef.importance == 0).sum(), "of", coef.shape[0], "coefficients are zero")
print("Top 10 most important variables:")
coef.iloc[0:10]

0 of 103 coefficients are zero
Top 10 most important variables:


,variable,importance
0,platelets,0.847785
1,age,0.400659
2,sex_male,0.081262
3,noise61,0.070293
4,noise19,0.050092
5,noise7,0.047130
6,noise0,0.045571
7,noise64,0.042336
8,noise25,0.041782
9,noise30,0.041497


## XGBoost

In [31]:
# Fit XGBoost with default settings on train+val data
# Like EBM, XGBoost does not require data to be scaled beforehand
clf = XGBClassifier()
clf.fit(x_sel, y_sel)

pred_sel = clf.predict_proba(x_sel)[:, 1]
pred_test = clf.predict_proba(x_test)[:, 1]

auc_sel = roc_auc_score(y_sel, pred_sel)
auc_test = roc_auc_score(y_test, pred_test)
print("auc train+val:", auc_sel)
print("auc test", auc_test)

auc train+val: 1.0
auc test 0.7705968894493486


In [32]:
# Look at feature importances
coef = clf.feature_importances_.reshape(1, -1)
coef = pd.DataFrame(coef, columns=x_train.columns)
coef = coef.transpose().reset_index()
coef.columns = ['variable', 'importance']
coef = coef.sort_values(by=['importance'], ascending=False).reset_index(drop=True)
print((coef.importance == 0).sum(), "of", coef.shape[0], "coefficients are zero")
print("Top 10 most important variables:")
coef.iloc[0:10]

0 of 103 coefficients are zero
Top 10 most important variables:


,variable,importance
0,platelets,0.045554
1,age,0.026832
2,noise47,0.023765
3,noise90,0.022720
4,noise19,0.016972
5,noise83,0.016332
6,noise61,0.015924
7,noise92,0.015854
8,noise30,0.015790
9,noise94,0.015747


In [33]:
# Use sequential model optimisation to find best hyperparameters for XGBoost
def objective_xgb(trial):
    params = {
        "max_depth": trial.suggest_int("max_depth", 2, 8),
        "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-5, 1e3, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-5, 1e3, log=True),
        "gamma": trial.suggest_float("gamma", 1e-3, 10, log=True),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 100),
        "subsample": trial.suggest_float("subsample", 0.5, 1),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1),
    }
    clf = XGBClassifier(**params, n_estimators=2000, eval_metric="auc", early_stopping_rounds=50)
    clf.fit(x_train, y_train, eval_set=[(x_val, y_val)], verbose=False)
    trial.set_user_attr("n_estimators", clf.best_iteration + 1)  # store the best number of trees
    return clf.best_score  # validation AUC at the best iteration

In [34]:
# Run the hyperparameter search
study_xgb = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
study_xgb.optimize(objective_xgb, n_trials=100)  # XGBoost runs quite fast so we use a higher number of trials here
best_n_estimators = study_xgb.best_trial.user_attrs["n_estimators"]
print("Best hyperparameters:", study_xgb.best_params)
print("Best number of trees:", best_n_estimators)
print("Best validation AUC:", study_xgb.best_value)

[I 2026-11-08 17:07:19,154] A new study created in memory with name: no-name-df3fbf8f-e1c3-4fe2-a728-1fe23b301682
[I 2026-11-08 17:07:19,332] Trial 0 finished with value: 0.812578814627995 and parameters: {'max_depth': 4, 'learning_rate': 0.22648248189516848, 'reg_alpha': 7.177141927992014, 'reg_lambda': 0.6155564318973021, 'gamma': 0.004207988669606638, 'min_child_weight': 16, 'subsample': 0.5290418060840998, 'colsample_bytree': 0.9330880728874675}. Best is trial 0 with value: 0.812578814627995.
[I 2026-11-08 17:07:19,531] Trial 1 finished with value: 0.7617171080285834 and parameters: {'max_depth': 6, 'learning_rate': 0.05675206026988748, 'reg_alpha': 1.4610865886287199e-05, 'reg_lambda': 574.4851636320429, 'gamma': 2.1368329072358767, 'min_child_weight': 22, 'subsample': 0.5909124836035503, 'colsample_bytree': 0.5917022549267169}. Best is trial 0 with value: 0.812578814627995.
[I 2026-11-08 17:07:19,931] Trial 2 finished with value: 0.8189365279529214 and parameters: {'max_depth': 4

Best hyperparameters: {'max_depth': 2, 'learning_rate': 0.1541271605050286, 'reg_alpha': 0.0006535563357210609, 'reg_lambda': 0.013151164968855841, 'gamma': 0.3783053809172756, 'min_child_weight': 4, 'subsample': 0.7407958231022965, 'colsample_bytree': 0.6792881788008427}
Best number of trees: 21
Best validation AUC: 0.8596048759983186


In [35]:
# Refit XGBoost with the best hyperparameters and number of trees on train + val set
# (no early stopping here, as there is no separate validation set left: the number of trees found above is used)
clf = XGBClassifier(**study_xgb.best_params, n_estimators=best_n_estimators)
clf.fit(x_sel, y_sel)

# Make predictions on training+validation and test data
pred_sel = clf.predict_proba(x_sel)[:, 1]
pred_test = clf.predict_proba(x_test)[:, 1]

# Evaluate
auc_sel = roc_auc_score(y_sel, pred_sel)
auc_test = roc_auc_score(y_test, pred_test)
print(f"auc sel: {auc_sel}, auc test: {auc_test}")

auc sel: 0.9036849269129861, auc test: 0.780369903320723


In [36]:
# Look at feature importances
coef = clf.feature_importances_.reshape(1, -1)
coef = pd.DataFrame(coef, columns=x_train.columns)
coef = coef.transpose().reset_index()
coef.columns = ['variable', 'importance']
coef = coef.sort_values(by=['importance'], ascending=False).reset_index(drop=True)
print((coef.importance == 0).sum(), "of", coef.shape[0], "coefficients are zero")
print("Top 10 most important variables:")
coef.iloc[0:10]

75 of 103 coefficients are zero
Top 10 most important variables:


,variable,importance
0,platelets,0.108989
1,age,0.071110
2,noise56,0.047947
3,noise13,0.046741
4,noise52,0.045332
5,noise72,0.044148
6,noise1,0.040593
7,noise7,0.036786
8,noise19,0.036333
9,noise0,0.035720
